# StateGen -- DS-1000 Experiments (Colab)

- **Runtime:** CPU only (no GPU needed)
- **Estimated time:** ~2-3 hrs for 200 tasks x 4 methods
- **Evaluation:** Real execution-based (not syntax check)
- **Dataset:** DS-1000 (100 Pandas + 100 Numpy = 200 tasks)

In [ ]:
!git clone https://github.com/Joshh99/StateGen.git
%cd StateGen
!pip install -r requirements.txt -q

In [ ]:
from google.colab import files
uploaded = files.upload()
import shutil
shutil.move(list(uploaded.keys())[0], ".env")

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()
key = os.getenv("TOGETHER_API_KEY")
print(f"TOGETHER_API_KEY: {'SET' if key else 'MISSING -- stop here'}")

## Configuration
Edit the variables below before running experiments.

In [ ]:
# Edit these as needed
PROVIDER = "together_ai"
MODEL = "deepseek-ai/DeepSeek-V3"
MAX_TASKS = 200        # 200 = full run (100 Pandas + 100 Numpy)
RESULTS_DIR = "results/ds1000"
# For ThetaEdge (future): set PROVIDER="openai_compatible" and
# add THETAEDGE_BASE_URL + THETAEDGE_API_KEY to your .env

In [ ]:
!python experiments/run_ds1000.py \
  --method direct_gen \
  --max_tasks 2 \
  --dry_run

In [ ]:
# Run this first to sanity check before full 200-task run
import subprocess
result = subprocess.run([
    "python", "experiments/run_ds1000.py",
    "--method", "direct_gen",
    "--max_tasks", "10",
    "--provider", PROVIDER,
    "--model", MODEL,
    "--results_dir", RESULTS_DIR
], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

## Full Experiment Run
Check pilot results above before proceeding.
Expected cost: ~$3-5 on Together.ai for full 200 tasks x 4 methods.

In [ ]:
# Run sequentially to avoid rate limits
import subprocess
METHODS = ["direct_gen", "self_planning", "self_debugging", "stategen"]
for method in METHODS:
    print(f"\n{'='*50}\nRunning: {method}\n{'='*50}")
    result = subprocess.run([
        "python", "experiments/run_ds1000.py",
        "--method", method,
        "--max_tasks", str(MAX_TASKS),
        "--provider", PROVIDER,
        "--model", MODEL,
        "--results_dir", RESULTS_DIR
    ], capture_output=True, text=True)
    print(result.stdout[-3000:])  # last 3000 chars to avoid cell overflow
    if result.returncode != 0:
        print("ERROR:", result.stderr[-1000:])
        break  # stop on first failure

In [ ]:
import json, os
metrics_path = f"{RESULTS_DIR}/metrics.json"
if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        m = json.load(f)
    print(json.dumps(m, indent=2))
else:
    print("No metrics file yet -- run experiments first")

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("ds1000_results", "zip", RESULTS_DIR)
files.download("ds1000_results.zip")